In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Download tables of results from **Metaphlan**  `abundance_table`  
Make a taxon an index  
Join tables by taxon  

In [ ]:
me_il = pd.read_csv("../data/merged_abundance_table_new.txt", sep="\t", skiprows=1)
me_he = pd.read_csv("../data/merged_abundance_table_he.txt", sep="\t", skiprows=1)
me_he = me_he.set_index("clade_name")
me_il = me_il.set_index('clade_name')
me_all = pd.concat([me_il, me_he], axis=1).fillna(0)

Create metadata with sample group 

In [ ]:
meta_il = pd.DataFrame({'sample': me_il.columns, 'group': 'disease'})
meta_he = pd.DataFrame({'sample': me_he.columns, 'group': 'healthy'})
meta = pd.concat([meta_il, meta_he], ignore_index=True)
meta = meta.set_index('sample')

Taxonomic level selection

In [ ]:
me_sp = me_all[me_all.index.str.contains(r"\|s__")].copy()
me_sp['taxon'] = me_sp.index.str.split('|').str[-1]
me_sp = me_sp.set_index('taxon')
me_sp.index = me_sp.index.str.replace('s__', '', regex=False)
me_sp = me_sp[~me_sp.index.str.startswith('t__')]
me_sp = me_sp[~me_sp.index.astype(str).str.contains("GGB", na=False)]

Group by healthy/disease.  
For average abundance by group

In [ ]:
me_t = me_sp.T
me_t["group"] = meta["group"]
grouped = me_t.groupby("group").mean()

Plot for Mean relative abundance

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

grouped.plot(kind="bar", stacked=True, ax=ax)

ax.set_ylabel("Mean relative abundance", fontsize=14)
ax.set_xlabel("Group", fontsize=14)

ax.tick_params(axis='x', labelsize=12, rotation=0)
ax.tick_params(axis='y', labelsize=12)

handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1],
    labels[::-1],
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=11,
    title="Taxa",
    title_fontsize=12
)

# plt.title('Microbial abundance Metaphlan', fontsize=20)

plt.tight_layout()
plt.savefig('mean_rel_abund100_metaphlan.png', dpi=300)
plt.show()

PCA

In [ ]:
from modules.pca import plot_pca

pca_df, pca = plot_pca(me_t, meta, method="Metaphlan")